<a href="https://colab.research.google.com/github/Shreshta1234/Fellowship-training-868661/blob/main/Phase%201/Shreshta_Nagaraju_Day_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip -q install kagglehub
import kagglehub, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

path = kagglehub.dataset_download("patelris/crop-yield-prediction-dataset")
df = pd.read_csv(os.path.join(path, "yield_df.csv"))

cols = ["average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp"]
df = df[cols + ["hg/ha_yield"]].dropna().sample(400, random_state=0)

rng = np.random.default_rng(0)
X_raw = df[cols].values.astype(float)
X_raw = X_raw + rng.normal(0, 0.3 * X_raw.std(axis=0), X_raw.shape)
y_raw = df["hg/ha_yield"].values / 10000


idx = rng.choice(len(y_raw), 8, replace=False)
y_raw[idx] *= 0.3
print(X_raw.shape, y_raw.mean())

Using Colab cache for faster access to the 'crop-yield-prediction-dataset' dataset.
(400, 3) 7.905314799999998


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

Xtr_r, Xte_r, ytr, yte = train_test_split(X_raw, y_raw, test_size=0.25, random_state=42)


prep = make_pipeline(PolynomialFeatures(3, include_bias=False), StandardScaler()).fit(Xtr_r)
Xtr, Xte = prep.transform(Xtr_r), prep.transform(Xte_r)


y_mean, y_std = ytr.mean(), ytr.std()
ytr_s, yte_s = (ytr - y_mean) / y_std, (yte - y_mean) / y_std
print("features after polynomial expansion:", Xtr.shape[1])

features after polynomial expansion: 19


In [3]:

def fit_ridge_gd(X, y, lam, lr=0.01, epochs=5000):
    n, d = X.shape
    w, b = np.zeros(d), 0.0
    for _ in range(epochs):
        err = X @ w + b - y
        w -= lr * (2 * X.T @ err / n + 2 * lam * w)
        b -= lr * 2 * err.mean()
    return w, b

def mse(X, y, w, b):
    return ((X @ w + b - y) ** 2).mean()

print(f"{'lambda':>8} {'train MSE':>10} {'test MSE':>10} {'||w||':>8}")
for lam in [0, 0.01, 0.1, 1, 10]:
    w, b = fit_ridge_gd(Xtr, ytr_s, lam)
    print(f"{lam:>8} {mse(Xtr,ytr_s,w,b):>10.4f} {mse(Xte,yte_s,w,b):>10.4f} {np.linalg.norm(w):>8.3f}")


from sklearn.linear_model import Ridge
lam = 0.1
w, b = fit_ridge_gd(Xtr, ytr_s, lam)
sk = Ridge(alpha=lam * len(Xtr)).fit(Xtr, ytr_s)
print("\nmax |w_manual - w_sklearn| =", np.abs(w - sk.coef_).max())

  lambda  train MSE   test MSE    ||w||
       0     0.9512     1.1509    0.856
    0.01     0.9549     1.1557    0.625
     0.1     0.9652     1.1742    0.263
       1     0.9828     1.1939    0.082
      10     0.9967     1.2045    0.012

max |w_manual - w_sklearn| = 1.501181121233569e-06


In [4]:

from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import LinearRegression

pipe = make_pipeline(PolynomialFeatures(3, include_bias=False), StandardScaler(), Ridge())
grid = {"ridge__alpha": np.logspace(-2, 4, 13)}
gs = GridSearchCV(pipe, grid, cv=KFold(5, shuffle=True, random_state=0),
                  scoring="neg_mean_squared_error", return_train_score=True).fit(Xtr_r, ytr)

print("Best alpha:", gs.best_params_["ridge__alpha"])


base = make_pipeline(PolynomialFeatures(3, include_bias=False), StandardScaler(),
                     LinearRegression()).fit(Xtr_r, ytr)
test_mse_base = ((base.predict(Xte_r) - yte) ** 2).mean()
test_mse_ridge = ((gs.predict(Xte_r) - yte) ** 2).mean()
print(f"Test MSE  unregularised: {test_mse_base:.4f}   tuned Ridge: {test_mse_ridge:.4f}")

Best alpha: 10000.0
Test MSE  unregularised: 75.0999   tuned Ridge: 79.7691
